Notes for self:

* State: what is traveling between the nodes of our graph. Encapsulates all information. For the agents, its going to be the:
- current instructions
- the previous implementation
- the set of code
- how many rounds are left(for adding in unncessary things after round 3)

Tools may be wrapped in a langchain setup `from langchain.tools import tool`
* binding the llm to the tool is letting the LLM actually make a call for the tool. But the ToolNode itself is what executes the actual tool code.

Router Node: When to go to tools node vs other states(or not tool nodes).
    grab the last message to see if a toolcall happened and theres text in the toolcall, then you return "tools"

# Module: LangChain & LangGraph

## What they are and why they exist

You just built **ChaosCodingAgents** with the raw Anthropic SDK. Two agents take turns rewriting code. The infrastructure you wrote by hand:
- `agents.py` — raw `client.messages.create()` calls, f-string context packaging, regex tag parsing
- `orchestrator.py` — an `Orchestrator` class managing the `for` loop, per-agent history lists, round counting
- `context_builder.py` — assembles the context string passed to each agent
- `context_trimmer.py` — trims history to stay under token limits

**LangChain** and **LangGraph** are the libraries that handle most of that plumbing for you.

| Library | Layer | What it replaces in CCA |
|---|---|---|
| **LangChain** | LLM abstraction | Raw `client.messages.create()`, f-string context packaging, regex output parsing |
| **LangGraph** | Orchestration | The `Orchestrator` class, the `for` loop, manual history lists, `context_trimmer.py` |

Think of it this way:
- **LangChain** = the toolkit for *individual* LLM interactions (calling a model, formatting prompts, parsing output)
- **LangGraph** = the framework for wiring those interactions into a stateful loop that knows whose turn it is and when to stop

---

## Key Vocabulary

| Term | Meaning | CCA equivalent |
|---|---|---|
| **ChatModel** | LangChain wrapper around an LLM (`ChatAnthropic`, `ChatOpenAI`). Unified interface. | `anthropic.Anthropic()` |
| **Message** | Typed object: `SystemMessage`, `HumanMessage`, `AIMessage`. | `{"role": "user", "content": "..."}` dicts |
| **ChatPromptTemplate** | Reusable, parameterized prompt builder. | `build_context_package()` |
| **LCEL** | LangChain Expression Language — a protocol that makes LangChain components **composable**. Every component (prompt, model, parser) implements the `Runnable` interface, exposing the same methods: `invoke`, `stream`, `batch`, `ainvoke`. The `\|` pipe operator wires them into a chain: `prompt \| model \| parser`. When you call `chain.invoke({...})`, each component's output becomes the next component's input. Streaming, async, and batching work on any chain for free — you don't wire them up manually. | Manual sequence: call `build_context_package()`, then `client.messages.create()`, then `_extract_code()` — three separate steps that can't be composed, streamed, or batched as a unit |
| **OutputParser** | Extracts structured data from model output. | `_extract_code()` / `_extract_critique()` regex |
| **StateGraph** | LangGraph class for defining a stateful graph with nodes and edges. | `Orchestrator` class |
| **State** | A `TypedDict` holding all data flowing through the graph. | `Orchestrator` instance variables |
| **Node** | A Python function `(state) -> dict` — one unit of work. | `edgeworth_turn()`, `light_turn()` |
| **Edge** | A connection between two nodes (always executes). | Sequential lines in the `for` loop |
| **Conditional Edge** | A routing function that reads state and returns the next node name. | `if round_num >= 3: ...` |
| **END** | Sentinel value — going here terminates the graph. | `break` / loop exhaustion |
| **Checkpointer** | Backend that saves graph state after every node. Free persistence. | `WorkspaceManager` + session files |


---
## How to Use This Notebook

1. **Run the setup cell** — verify packages are installed and your API key is set.
2. **Part 1 (LangChain)** covers individual LLM interactions. Most cells require `ANTHROPIC_API_KEY`.
3. **Part 2 (LangGraph)** covers graph orchestration. The structural exercises (Ex 4–6) run without an API key — they use placeholder functions instead of LLM calls, so you can learn the graph mechanics first.
4. **Part 3 (CCA → LangGraph)** shows the full translation from the hand-rolled project to a LangGraph implementation. Requires `ANTHROPIC_API_KEY`.

Each exercise includes a **CCA reference** in comments — the specific file and line being replaced.


In [1]:
# Dependencies are already in environment.yml. Run once from your terminal:
#   mamba env create -f environment.yml    # first time
#   mamba env update -f environment.yml    # to pick up new packages
#   mamba activate chaos-agents
#
# If this cell reports MISSING packages, your environment is not active.
import os

missing = []
for pkg, name in [('langchain_anthropic', 'langchain-anthropic'), ('langchain_core', 'langchain-core')]:
    try:
        mod = __import__(pkg)
        print(f'  OK  {name} {mod.__version__}')
    except ImportError:
        missing.append(name)
        print(f'  MISSING  {name}')

if missing:
    print(f'\nEnvironment not active or packages missing.')
    print(f'Run: mamba activate chaos-agents')
    print(f'Or add to environment.yml and run: mamba env update -f environment.yml')

print()
if os.getenv('ANTHROPIC_API_KEY'):
    print('  ANTHROPIC_API_KEY is set -- LLM exercises will run')
else:
    print('  ANTHROPIC_API_KEY not set')
    print('    export ANTHROPIC_API_KEY=your_key_here')
    print('    Part 2 structural exercises run without it')


  OK  langchain-anthropic 1.4.7
  OK  langchain-core 1.4.8

  ANTHROPIC_API_KEY is set -- LLM exercises will run


---
---
# Part 1: LangChain
## From Raw SDK Calls to Composable Chains

LangChain wraps LLM providers behind a unified interface and gives you building blocks for prompts, chains, and output parsing. The individual pieces are small — the value is in how they compose.

```
ChatPromptTemplate  →  ChatAnthropic  →  OutputParser
       ↕                   ↕                  ↕
 build_context_package()  client.messages.create()  _extract_code() / _extract_critique()
 [CCA context_builder.py]  [CCA agents.py]  [CCA agents.py regex]
```


---
## Exercise 1: Raw Anthropic SDK vs LangChain

CCA makes LLM calls in `agents.py` with the raw `anthropic` client. The pattern:
```python
resp = client.messages.create(model=..., max_tokens=..., system=[...], messages=[...])
text = resp.content[0].text
```

LangChain wraps this. The key changes:
- `system=[{"type": "text", "text": ...}]` → `SystemMessage(content=...)` object
- `messages=[{"role": "user", "content": ...}]` → `HumanMessage(content=...)` object
- `resp.content[0].text` → `response.content` (the response IS an `AIMessage`)

Both call the exact same Claude API. LangChain is a typed wrapper, not a different backend.


In [2]:
# ── BEFORE: What CCA does in agents.py _call_agent() ─────────────────────────
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
raw_resp = client.messages.create(
    model='claude-haiku-4-5-20251001',
    max_tokens=128,
    system=[{'type': 'text', 'text': 'You are Edgeworth. Be cold and concise.'}],
    messages=[{'role': 'user', 'content': 'What is a strategy pattern? One sentence.'}],
)
raw_text = raw_resp.content[0].text
print('RAW SDK result type:', type(raw_resp.content[0]))
print('Text:', raw_text)
print()

# ── AFTER: LangChain equivalent ───────────────────────────────────────────────
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

model = ChatAnthropic(model='claude-haiku-4-5-20251001', max_tokens=128)
lc_resp = model.invoke([
    SystemMessage(content='You are Edgeworth. Be cold and concise.'),
    HumanMessage(content='What is a strategy pattern? One sentence.'),
])
print('LangChain result type:', type(lc_resp))
print('Text:', lc_resp.content)
print()

print('Key mapping:')
print('  resp.content[0].text  →  response.content  (AIMessage.content is the text)')
print('  {"role": "user", ...}  →  HumanMessage(content=...)')
print('  {"role": "assistant", ...}  →  AIMessage(content=...)')
print('  system=[{"type": "text", ...}]  →  SystemMessage(content=...)')


RAW SDK result type: <class 'anthropic.types.text_block.TextBlock'>
Text: The Strategy pattern defines a family of interchangeable algorithms, encapsulating each one so the client can select the appropriate strategy at runtime.

LangChain result type: <class 'langchain_core.messages.ai.AIMessage'>
Text: The Strategy pattern defines a family of interchangeable algorithms, encapsulates each one, and lets the client select which one to use at runtime.

Key mapping:
  resp.content[0].text  →  response.content  (AIMessage.content is the text)
  {"role": "user", ...}  →  HumanMessage(content=...)
  {"role": "assistant", ...}  →  AIMessage(content=...)
  system=[{"type": "text", ...}]  →  SystemMessage(content=...)


---
## Exercise 2: Prompt Templates

CCA assembles its context string in `context_builder.py` with a multi-line f-string:
```python
return (
    f"FEATURE REQUEST:\n{feature_request}\n\n"
    f"CURRENT CODEBASE:\n{codebase_str}\n\n"
    f"PREVIOUS AGENT'S CRITIQUE:\n{critique_selection}\n\n"
    f"ROUND: {round_num}\n"
    f"YOUR TURN: {agent_name}"
)
```

`ChatPromptTemplate` is the declarative equivalent. The template is **defined once** and **filled at call time** — the variables are explicit, the structure is reusable, and you can unit-test it without an LLM.


In [3]:
from langchain_core.prompts import ChatPromptTemplate

# Replaces build_context_package() in context_builder.py
CONTEXT_TEMPLATE = ChatPromptTemplate.from_messages([
    ('system', '{system_prompt}'),
    ('human', (
        'FEATURE REQUEST:\n{feature_request}\n\n'
        'CURRENT CODEBASE:\n{codebase}\n\n'
        'PREVIOUS CRITIQUE:\n{previous_critique}\n\n'
        'ROUND: {round_num}\n'
        'YOUR TURN: {agent_name}'
    )),
])

# format_messages() fills in the variables and returns a list of typed message objects.
# No API call yet — this is pure string formatting.
messages = CONTEXT_TEMPLATE.format_messages(
    system_prompt='You are Edgeworth. Cold, precise, condescending.',
    feature_request='Filter prime numbers from a list.',
    codebase='# solution.py\n# empty — round 1',
    previous_critique='None - this is round 1. Write your first implementation.',
    round_num=1,
    agent_name='EDGEWORTH',
)

print(f'Formatted {len(messages)} messages:')
for msg in messages:
    print(f'\n  [{msg.__class__.__name__}]')
    print(f'  {repr(msg.content[:150])}...')

print()
print('Advantages over the CCA f-string approach:')
print('  - Variables are explicit and validated (missing key raises an error immediately)')
print('  - Template is separable from the call — easier to test and reuse')
print('  - Works directly with LCEL (the | pipe operator, see Exercise 3)')


Formatted 2 messages:

  [SystemMessage]
  'You are Edgeworth. Cold, precise, condescending.'...

  [HumanMessage]
  'FEATURE REQUEST:\nFilter prime numbers from a list.\n\nCURRENT CODEBASE:\n# solution.py\n# empty — round 1\n\nPREVIOUS CRITIQUE:\nNone - this is round 1. Writ'...

Advantages over the CCA f-string approach:
  - Variables are explicit and validated (missing key raises an error immediately)
  - Template is separable from the call — easier to test and reuse
  - Works directly with LCEL (the | pipe operator, see Exercise 3)


---
## Exercise 3: LCEL Chains and Structured Output

### The `|` pipe operator
LCEL (LangChain Expression Language) lets you build a **chain** by piping outputs into inputs:
```
chain = prompt | model | parser
result = chain.invoke({"feature": "...", "code": "..."})
```
This is what a single agent turn does in CCA — build context, call model, parse output — but expressed declaratively in one line.

### Structured Output — replaces CCA's regex parser
CCA parses `<code>` and `<critique>` tags with regex in `_extract_code()` and `_extract_critique()`. `model.with_structured_output(PydanticSchema)` asks the model to return a typed object directly — no regex needed.


In [4]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field

model = ChatAnthropic(model='claude-haiku-4-5-20251001', max_tokens=512)

# ── Part A: LCEL chain with StrOutputParser ───────────────────────────────────
# Returns the model's response as a plain string.
simple_chain = (
    ChatPromptTemplate.from_messages([
        ('system', 'You are Edgeworth. Rewrite the code. Add a brief condescending critique.'),
        ('human', 'Feature: {feature}\nCurrent code:\n{code}'),
    ])
    | model
    | StrOutputParser()   # extracts AIMessage.content as a plain string
)

raw_result = simple_chain.invoke({
    'feature': 'filter primes from a list',
    'code': '# empty — first round',
})
print('LCEL chain result (plain text):')
print(raw_result[:300])
print()

# ── Part B: Structured Output — replaces regex in agents.py ──────────────────
# CCA uses _extract_code() + _extract_critique() with regex on XML tags.
# with_structured_output() asks the model to return a typed Pydantic object.

class AgentResponse(BaseModel):
    code: str = Field(description='Complete Python implementation')
    critique: str = Field(description='2-4 sentences of in-character critique of the previous implementation')

structured_chain = (
    ChatPromptTemplate.from_messages([
        ('system', 'You are Edgeworth. Critique and rewrite the code. Be condescending.'),
        ('human', 'Feature: {feature}\nCurrent code:\n{code}'),
    ])
    | model.with_structured_output(AgentResponse)
    # no output parser needed — the model returns an AgentResponse object
)

structured_result = structured_chain.invoke({
    'feature': 'filter primes from a list',
    'code': 'def solve(nums): return [n for n in nums if n > 1]',
})

print('Structured output (no regex needed):')
print(f'  type: {type(structured_result).__name__}')
print(f'  .code: {structured_result.code[:120]}...')
print(f'  .critique: {structured_result.critique}')
print()
print('Replaces in CCA:')
print('  _extract_code(text, "code")     →  response.code')
print('  _extract_critique(text)          →  response.critique')


LCEL chain result (plain text):
# OBJECTION! That code is utterly lacking in substance.

```python
def filter_primes(numbers):
    """Filter a list to return only prime numbers."""
    def is_prime(n):
        if n < 2:
            return False
        if n == 2:
            return True
        if n % 2 == 0:
            return Fa

Structured output (no regex needed):
  type: AgentResponse
  .code: 
def solve(nums):
    def is_prime(n):
        if n < 2:
            return False
        if n == 2:
            return ...
  .critique: Your implementation is woefully inadequate. Filtering for numbers greater than 1 is not filtering for primes—that would include every composite number as well. A proper solution requires a primality test that actually checks divisibility, something you've apparently overlooked entirely. This is elementary mathematics.

Replaces in CCA:
  _extract_code(text, "code")     →  response.code
  _extract_critique(text)          →  response.critique


---
---
# Part 2: LangGraph
## Stateful, Cyclic Multi-Agent Workflows

LangChain handles individual LLM calls well. But CCA's core challenge isn't *calling* the LLM — it's *coordinating* multiple agents in a loop, tracking state across turns, and deciding when to stop. That's what LangGraph is for.

**The mental model:** your workflow is a directed graph.
- **Nodes** are functions that do work and update state.
- **Edges** decide what runs next.
- **State** is a dictionary that flows through every node — every node reads it and returns a partial update.
- **Cycles** are allowed — you can loop back to a previous node.

```
                    ┌─────────────┐
     ┌──────────────│  edgeworth  │◄──────────────┐
     │              └─────────────┘               │
     │ (always)            │ (always)             │  (if round <= max)
     │              ┌──────▼──────┐               │
  [START]           │    light    │───────────────►┘
                    └─────────────┘
                           │
                           │ (if round > max)
                    ┌──────▼──────┐
                    │   intern    │──► [END]
                    └─────────────┘
```

This is the CCA loop expressed as a graph. The `for` loop, the `if rounds_completed >= max` check, and the `summon_intern()` call all collapse into graph edges.


---
## Exercise 4: Your First LangGraph

The minimal graph: two nodes, one edge. No LLM. This runs without an API key.

**Node contract:** every node is a function `(state: State) -> dict`. The returned dict is a **partial state update** — only the keys you return are changed. Unreturned keys stay as they were.


In [5]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# ── 1. Define the state schema ────────────────────────────────────────────────
# TypedDict tells LangGraph what keys are valid and what types they hold.
class SimpleState(TypedDict):
    message: str

# ── 2. Define nodes (plain Python functions) ──────────────────────────────────
def node_a(state: SimpleState) -> dict:
    print(f'[Node A] received: {state["message"]}')
    return {'message': state['message'] + ' -> processed by A'}

def node_b(state: SimpleState) -> dict:
    print(f'[Node B] received: {state["message"]}')
    return {'message': state['message'] + ' -> processed by B'}

# ── 3. Build the graph ────────────────────────────────────────────────────────
graph = StateGraph(SimpleState)
graph.add_node('node_a', node_a)
graph.add_node('node_b', node_b)
graph.set_entry_point('node_a')       # execution starts here
graph.add_edge('node_a', 'node_b')    # after node_a, always go to node_b
graph.add_edge('node_b', END)         # after node_b, stop

# ── 4. Compile and run ────────────────────────────────────────────────────────
app = graph.compile()
result = app.invoke({'message': 'start'})
print(f'\nFinal state: {result}')
print()
print('Concepts:')
print('  StateGraph(Schema) -- defines graph with a TypedDict schema')
print('  add_node(name, fn) -- registers a function as a node')
print('  set_entry_point()  -- where execution starts')
print('  add_edge(a, b)     -- always: after a, go to b')
print('  END                -- sentinel: going here terminates the graph')
print('  compile()          -- locks the graph, returns a runnable Runnable')
print('  invoke(state)      -- runs the graph, returns final state')


[Node A] received: start
[Node B] received: start -> processed by A

Final state: {'message': 'start -> processed by A -> processed by B'}

Concepts:
  StateGraph(Schema) -- defines graph with a TypedDict schema
  add_node(name, fn) -- registers a function as a node
  set_entry_point()  -- where execution starts
  add_edge(a, b)     -- always: after a, go to b
  END                -- sentinel: going here terminates the graph
  compile()          -- locks the graph, returns a runnable Runnable
  invoke(state)      -- runs the graph, returns final state


---
## Exercise 5: State + Cycles — The Core CCA Pattern

The real CCA loop is cyclic: Edgeworth → Light → Edgeworth → Light → ... → Intern.

Two new concepts:
1. **Annotated reducer** — `Annotated[list[str], operator.add]` tells LangGraph to *append* new items to the list rather than replace it. This is how CCA would accumulate all critiques across rounds.
2. **Conditional edge** — a routing function reads state and returns the name of the next node. This replaces the `for` loop termination condition in `orchestrator.py`.

No LLM needed — nodes are placeholders. Focus on the loop mechanics.


In [6]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated, Optional
import operator

class LoopState(TypedDict):
    feature_request: str
    code: str
    round: int
    max_rounds: int
    last_critique: Optional[str]
    critiques: Annotated[list[str], operator.add]  # reducer: new items are appended

def edgeworth_turn(state: LoopState) -> dict:
    r = state['round']
    print(f'  [EDGEWORTH] Round {r}')
    new_code = f'# Edgeworth round {r}: architectural perfection'
    critique = f'The previous implementation was an embarrassment. Round {r} corrected.'
    # Note: 'round' is NOT returned here -- it stays unchanged.
    # Only light_turn increments the round counter.
    return {
        'code': new_code,
        'last_critique': critique,
        'critiques': [f'EDGEWORTH R{r}: {critique}'],  # appended via reducer
    }

def light_turn(state: LoopState) -> dict:
    r = state['round']
    print(f'  [LIGHT] Round {r}')
    new_code = f'# Light round {r}: divine inevitability'
    critique = f'Edgeworth sees structure. I see destiny. Round {r} is complete.'
    return {
        'code': new_code,
        'last_critique': critique,
        'critiques': [f'LIGHT R{r}: {critique}'],  # appended via reducer
        'round': r + 1,  # only light increments -- Light ends each full round
    }

def intern_summary(state: LoopState) -> dict:
    print(f'\n  [INTERN] Oh god oh god you\'re back --')
    print(f'  You asked for: {state["feature_request"]}')
    print(f'  {state["max_rounds"]} rounds completed. {len(state["critiques"])} total critiques.')
    return {}

# ── Routing function: replaces the for loop condition in orchestrator.py ───────
def should_continue(state: LoopState) -> str:
    if state['round'] > state['max_rounds']:
        return 'intern'
    return 'edgeworth'

# ── Build ─────────────────────────────────────────────────────────────────────
graph = StateGraph(LoopState)
graph.add_node('edgeworth', edgeworth_turn)
graph.add_node('light', light_turn)
graph.add_node('intern', intern_summary)

graph.set_entry_point('edgeworth')
graph.add_edge('edgeworth', 'light')            # always: edgeworth → light
graph.add_conditional_edges(                    # light → ? (routing function decides)
    'light',
    should_continue,
    {'edgeworth': 'edgeworth', 'intern': 'intern'},  # map return values to node names
)
graph.add_edge('intern', END)

app = graph.compile()

print('Running CCA loop (3 rounds, no LLM):')
result = app.invoke({
    'feature_request': 'Filter prime numbers from a list',
    'code': '',
    'round': 1,
    'max_rounds': 3,
    'last_critique': None,
    'critiques': [],
})

print(f'\nFinal code: {result["code"]}')
print(f'\nAll {len(result["critiques"])} accumulated critiques:')
for c in result['critiques']:
    print(f'  {c}')


Running CCA loop (3 rounds, no LLM):
  [EDGEWORTH] Round 1
  [LIGHT] Round 1
  [EDGEWORTH] Round 2
  [LIGHT] Round 2
  [EDGEWORTH] Round 3
  [LIGHT] Round 3

  [INTERN] Oh god oh god you're back --
  You asked for: Filter prime numbers from a list
  3 rounds completed. 6 total critiques.

Final code: # Light round 3: divine inevitability

All 6 accumulated critiques:
  EDGEWORTH R1: The previous implementation was an embarrassment. Round 1 corrected.
  LIGHT R1: Edgeworth sees structure. I see destiny. Round 1 is complete.
  EDGEWORTH R2: The previous implementation was an embarrassment. Round 2 corrected.
  LIGHT R2: Edgeworth sees structure. I see destiny. Round 2 is complete.
  EDGEWORTH R3: The previous implementation was an embarrassment. Round 3 corrected.
  LIGHT R3: Edgeworth sees structure. I see destiny. Round 3 is complete.


---
## Exercise 6: Conditional Edges in Depth — The Drift Directive

CCA's drift directive (`agents.py`, `_build_system()`):
```python
def _build_system(base: str, drift: str, round_num: int) -> list[dict]:
    text = base + (drift if round_num >= 3 else "")
```

In the hand-rolled version, this check is inside the agent function. In LangGraph, this becomes a **conditional edge** — a routing function that reads state and returns a node name string. This separates the *routing logic* from the *agent logic*.

The routing function is a pure function of state: `(state) -> str`. No side effects. No LLM call. Just a decision.


In [7]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

class DriftState(TypedDict):
    round: int
    mode: str
    output: str

def prepare_turn(state: DriftState) -> dict:
    print(f'  [EDGEWORTH] Round {state["round"]} -- selecting mode...')
    return {}

def normal_turn(state: DriftState) -> dict:
    print('    -> Normal: following the spec exactly')
    return {
        'mode': 'normal',
        'output': f'Round {state["round"]}: clean implementation as requested',
    }

def drift_turn(state: DriftState) -> dict:
    print('    -> DRIFT: adding unrequested features because I know better')
    return {
        'mode': 'drift',
        'output': f'Round {state["round"]}: implementation + strategy + factory + result type',
    }

def check_drift(state: DriftState) -> str:
    """Routing function: pure function of state, returns node name."""
    # Replaces: if round_num >= 3: text = base + drift
    return 'drift' if state['round'] >= 3 else 'normal'

graph = StateGraph(DriftState)
graph.add_node('prepare', prepare_turn)
graph.add_node('normal', normal_turn)
graph.add_node('drift', drift_turn)

graph.set_entry_point('prepare')
graph.add_conditional_edges(
    'prepare',
    check_drift,
    {'normal': 'normal', 'drift': 'drift'},
)
graph.add_edge('normal', END)
graph.add_edge('drift', END)

app = graph.compile()

print('Testing drift directive routing:')
for r in [1, 2, 3, 5]:
    result = app.invoke({'round': r, 'mode': 'normal', 'output': ''})
    print(f'  Round {r}: mode={result["mode"]}')
    print(f'    {result["output"]}')
    print()

print('In the full CCA graph, this routing would happen INSIDE the edgeworth/light nodes')
print('rather than as a separate edge. The point: routing logic is separable from node logic.')


Testing drift directive routing:
  [EDGEWORTH] Round 1 -- selecting mode...
    -> Normal: following the spec exactly
  Round 1: mode=normal
    Round 1: clean implementation as requested

  [EDGEWORTH] Round 2 -- selecting mode...
    -> Normal: following the spec exactly
  Round 2: mode=normal
    Round 2: clean implementation as requested

  [EDGEWORTH] Round 3 -- selecting mode...
    -> DRIFT: adding unrequested features because I know better
  Round 3: mode=drift
    Round 3: implementation + strategy + factory + result type

  [EDGEWORTH] Round 5 -- selecting mode...
    -> DRIFT: adding unrequested features because I know better
  Round 5: mode=drift
    Round 5: implementation + strategy + factory + result type

In the full CCA graph, this routing would happen INSIDE the edgeworth/light nodes
rather than as a separate edge. The point: routing logic is separable from node logic.


---
---
# Part 3: ChaosCodingAgents → LangGraph
## The Full Translation

Now we map the entire CCA project onto LangGraph abstractions.

| CCA (hand-rolled) | LangGraph equivalent |
|---|---|
| `Orchestrator.__init__` instance variables | `CCAState` TypedDict |
| `self.edgeworth_history: list[dict]` | `edgeworth_history: Annotated[list[BaseMessage], add_messages]` |
| `self.light_history: list[dict]` | `light_history: Annotated[list[BaseMessage], add_messages]` |
| `self.last_critique: str` | `state["last_critique"]` |
| `self.rounds_completed: int` | `state["round"]` |
| `Orchestrator._run_real_loop()` for loop | Graph cycle: `light → edgeworth` conditional edge |
| `if round_num >= 3:` drift check | `should_continue()` routing function or inline in node |
| `edgeworth_turn()` function | `edgeworth_node` graph node |
| `light_turn()` function | `light_node` graph node |
| `intern_summary()` function | `intern_node` graph node |
| `orchestrator.summon_intern()` call | Graph edge: `light → intern` when rounds exhausted |
| `context_trimmer.py` manual trimming | `add_messages` reducer + optional `trim_messages()` |
| `WorkspaceManager` disk writes | Side effect inside nodes (keep as-is, or use checkpointer) |
| `run_feedback_mode()` interactive loop | `interrupt_before=["light"]` + human-in-the-loop API |


---
## Exercise 7: Define the State Schema

The `Orchestrator` class disappears. Its instance variables become the state `TypedDict`. The graph carries this dict through every node transition.

The two history lists are the most important field to get right. CCA stores them as `list[dict]` with raw `{"role": ..., "content": ...}` dicts. In LangGraph the idiomatic type is `list[BaseMessage]`, and the `add_messages` reducer handles appending (and optionally trimming) automatically.


In [8]:
from typing import TypedDict, Annotated, Optional
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

class CCAState(TypedDict):
    # ── Immutable inputs ──────────────────────────────────────────────────
    # Set at invoke() time. Never overwritten by any node.
    feature_request: str         # replaces: self.feature_request
    max_rounds: int              # replaces: self.rounds

    # ── Turn tracking ─────────────────────────────────────────────────────
    # Plain values -- returning a new value from a node replaces the old one.
    round: int                   # replaces: self.rounds_completed

    # ── Shared workspace ──────────────────────────────────────────────────
    code: str                    # latest code version (Light always overwrites Edgeworth's)
    last_critique: Optional[str] # replaces: self.last_critique

    # ── Per-agent conversation histories ──────────────────────────────────
    # Annotated[list[BaseMessage], add_messages]:
    #   - TypedDict says this field holds a list of messages
    #   - add_messages is the REDUCER: when a node returns new messages,
    #     LangGraph calls add_messages(existing_list, new_messages) to merge them.
    #     This APPENDS new messages rather than replacing the whole list.
    #
    # Replaces: self.edgeworth_history = [] and self.light_history = []
    # Replaces: context_trimmer.py (add_messages has built-in trim support)
    edgeworth_history: Annotated[list[BaseMessage], add_messages]
    light_history: Annotated[list[BaseMessage], add_messages]

print('CCAState defined -- the Orchestrator class is replaced by this TypedDict.')
print()
print('About add_messages:')
print('  When edgeworth_node returns {"edgeworth_history": [msg1, msg2]},')
print('  LangGraph does: existing_history + [msg1, msg2]  (append, not replace)')
print('  This is equivalent to: self.edgeworth_history.append(msg1, msg2)')
print()
print('For trimming: langgraph has trim_messages() that integrates with add_messages.')
print('  This replaces context_trimmer.py entirely.')


CCAState defined -- the Orchestrator class is replaced by this TypedDict.

About add_messages:
  When edgeworth_node returns {"edgeworth_history": [msg1, msg2]},
  LangGraph does: existing_history + [msg1, msg2]  (append, not replace)
  This is equivalent to: self.edgeworth_history.append(msg1, msg2)

For trimming: langgraph has trim_messages() that integrates with add_messages.
  This replaces context_trimmer.py entirely.


---
## Exercise 8: Define the Nodes

Each agent function from `agents.py` becomes a LangGraph node. The node:
1. Reads context from `state`
2. Makes the LLM call (using `context_builder.py` as-is — no changes needed)
3. Returns a **partial state update** dict

Key difference from CCA: the node doesn't manage history manually. It just returns `{"edgeworth_history": [new_user_msg, ai_response]}` and the `add_messages` reducer appends those to the existing history.


In [9]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))  # so we can import from ChaosCodingAgents/

import re
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage

# Reuse CCA's context builder as-is -- it produces a string, and we wrap it in HumanMessage
from context_builder import build_context_package
from agents import EDGEWORTH_SYSTEM, EDGEWORTH_DRIFT, LIGHT_SYSTEM, LIGHT_DRIFT

AGENT_MODEL = 'claude-sonnet-4-6'
INTERN_MODEL = 'claude-haiku-4-5-20251001'

def _extract(text: str, tag: str) -> str:
    m = re.search(rf'<{tag}>(.*?)</{tag}>', text, re.DOTALL)
    return m.group(1).strip() if m else ''

def edgeworth_node(state: CCAState) -> dict:
    r = state['round']
    system_text = EDGEWORTH_SYSTEM + (EDGEWORTH_DRIFT if r >= 3 else '')

    pkg = build_context_package(
        feature_request=state['feature_request'],
        codebase={'solution.py': state.get('code', '')},
        previous_critique=state.get('last_critique'),
        agent_name='EDGEWORTH',
        round_num=r,
    )
    new_user_msg = HumanMessage(content=pkg)

    # Build message list: system + existing history + new user message
    # Note: system is passed separately; history only contains human/AI turns
    all_msgs = [SystemMessage(content=system_text)] + list(state.get('edgeworth_history', [])) + [new_user_msg]

    model = ChatAnthropic(model=AGENT_MODEL, max_tokens=8192)
    response = model.invoke(all_msgs)  # response is AIMessage

    code = _extract(response.content, 'code')
    critique = _extract(response.content, 'critique')

    print(f'[EDGEWORTH] Round {r} -- critique: {critique[:80]}...')
    return {
        'code': code,
        'last_critique': critique,
        'edgeworth_history': [new_user_msg, response],  # add_messages APPENDS these
    }

def light_node(state: CCAState) -> dict:
    r = state['round']
    system_text = LIGHT_SYSTEM + (LIGHT_DRIFT if r >= 3 else '')

    pkg = build_context_package(
        feature_request=state['feature_request'],
        codebase={'solution.py': state.get('code', '')},
        previous_critique=state.get('last_critique'),
        agent_name='LIGHT',
        round_num=r,
    )
    new_user_msg = HumanMessage(content=pkg)
    all_msgs = [SystemMessage(content=system_text)] + list(state.get('light_history', [])) + [new_user_msg]

    model = ChatAnthropic(model=AGENT_MODEL, max_tokens=8192)
    response = model.invoke(all_msgs)

    code = _extract(response.content, 'code')
    critique = _extract(response.content, 'critique')

    print(f'[LIGHT] Round {r} -- critique: {critique[:80]}...')
    return {
        'code': code,
        'last_critique': critique,
        'light_history': [new_user_msg, response],  # add_messages APPENDS these
        'round': r + 1,   # only Light increments -- Light ends the round
    }

def intern_node(state: CCAState) -> dict:
    print('\n[INTERN] Oh god oh god you\'re back --')
    model = ChatAnthropic(model=INTERN_MODEL, max_tokens=2048)
    from agents import INTERN_SYSTEM
    response = model.invoke([
        SystemMessage(content=INTERN_SYSTEM),
        HumanMessage(content=f'ORIGINAL FEATURE REQUEST:\n{state["feature_request"]}\n\nFINAL CODE:\n{state.get("code", "")}'),
    ])
    print(response.content)
    return {}

def should_continue(state: CCAState) -> str:
    # Replaces: the for loop condition + orchestrator.summon_intern() trigger
    if state['round'] > state['max_rounds']:
        return 'intern'
    return 'edgeworth'

print('Nodes defined: edgeworth_node, light_node, intern_node, should_continue')
print()
print('What changed vs CCA agents.py:')
print('  - No global history management -- node receives state, returns update')
print('  - add_messages reducer handles history accumulation automatically')
print('  - build_context_package() reused as-is (no changes to context_builder.py)')
print('  - System prompts reused from agents.py (no changes)')


Nodes defined: edgeworth_node, light_node, intern_node, should_continue

What changed vs CCA agents.py:
  - No global history management -- node receives state, returns update
  - add_messages reducer handles history accumulation automatically
  - build_context_package() reused as-is (no changes to context_builder.py)
  - System prompts reused from agents.py (no changes)


---
## Exercise 9: Assemble and Run the Graph

The graph wiring is 8 lines. Everything else is already defined — the nodes, the routing function, the state schema.


In [10]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

# ── Build the graph ───────────────────────────────────────────────────────────
graph = StateGraph(CCAState)

graph.add_node('edgeworth', edgeworth_node)
graph.add_node('light', light_node)
graph.add_node('intern', intern_node)

graph.set_entry_point('edgeworth')
graph.add_edge('edgeworth', 'light')
graph.add_conditional_edges('light', should_continue, {
    'edgeworth': 'edgeworth',
    'intern': 'intern',
})
graph.add_edge('intern', END)

# ── Compile with checkpointer ─────────────────────────────────────────────────
# MemorySaver saves graph state after every node -- free session persistence.
# The thread_id in config identifies the session.
checkpointer = MemorySaver()
cca_app = graph.compile(checkpointer=checkpointer)

# ── (Optional) Visualize ──────────────────────────────────────────────────────
# from IPython.display import Image, display
# try:
#     display(Image(cca_app.get_graph().draw_mermaid_png()))
# except Exception:
#     print(cca_app.get_graph().draw_ascii())

print('Graph compiled.')
print()
print('Structure:')
print('  START --> edgeworth')
print('  edgeworth --> light  (always)')
print('  light --> edgeworth  (if round <= max_rounds)')
print('  light --> intern     (if round > max_rounds)')
print('  intern --> END')
print()

INITIAL_STATE = {
    'feature_request': 'Write a function that filters prime numbers from a list',
    'code': '',
    'round': 1,
    'max_rounds': 3,
    'last_critique': None,
    'edgeworth_history': [],
    'light_history': [],
}

# ── Invoke (requires ANTHROPIC_API_KEY) ───────────────────────────────────────
# Uncomment to run:
#
config = {'configurable': {'thread_id': 'cca-session-001'}}
result = cca_app.invoke(INITIAL_STATE, config=config)
print('\nFinal code:')
print(result['code'])

# ── Stream mode: watch each turn as it completes ──────────────────────────────
# for chunk in cca_app.stream(INITIAL_STATE, stream_mode='updates'):
#     node_name = list(chunk.keys())[0]
#     delta = chunk[node_name]
#     print(f'[{node_name.upper()}] updated keys: {list(delta.keys())}')

print('To run: set ANTHROPIC_API_KEY and uncomment invoke() or stream() above.')


Graph compiled.

Structure:
  START --> edgeworth
  edgeworth --> light  (always)
  light --> edgeworth  (if round <= max_rounds)
  light --> intern     (if round > max_rounds)
  intern --> END

[EDGEWORTH] Round 1 -- critique: The codebase was entirely absent — a blank file, which speaks volumes about the ...
[LIGHT] Round 1 -- critique: The previous agent arrived with trial division and called it complete — a man wh...
[EDGEWORTH] Round 2 -- critique: The previous agent's `filter_primes_with_stats` returned a raw dictionary — a st...
[LIGHT] Round 2 -- critique: The previous agent replaced my dictionary with a class and called it enlightenme...
[EDGEWORTH] Round 3 -- critique: The previous agent's `PrimeFilterStats` used a hand-rolled `_sorted` cache via a...
[LIGHT] Round 3 -- critique: The previous implementation capped its ambition at one million — a ceiling so ar...

[INTERN] Oh god oh god you're back --
*looks up frantically from screen, eyes wide*

You asked for: A function tha

---
---
# Bonus: What You Get For Free

The hand-rolled CCA implementation works. But LangGraph gives you several capabilities for free that CCA would need significant additional code to replicate.

## 1. Checkpointing — Free Session Persistence

CCA uses `WorkspaceManager` to write `solution.py` to disk per session. LangGraph's checkpointer saves the *entire graph state* after every node — not just the code file.

```python
from langgraph.checkpoint.sqlite import SqliteSaver

checkpointer = SqliteSaver.from_conn_string('cca_sessions.db')
app = graph.compile(checkpointer=checkpointer)

# Every node's state snapshot is saved automatically.
# Resume a session by passing the same thread_id:
config = {'configurable': {'thread_id': 'session-001'}}
result = app.invoke(INITIAL_STATE, config=config)

# Or inspect past state:
history = list(app.get_state_history(config))
# history is a list of StateSnapshot objects — one per node transition
```

## 2. Human-in-the-Loop — Replaces `run_feedback_mode()`

CCA's `feedback.py` implements a full interactive loop where the user can comment on what the agents built. LangGraph has this built in: `interrupt_before` pauses the graph before a node runs and waits for human input.

```python
# Pause before Light runs so a human can inspect/edit Edgeworth's code
app = graph.compile(
    checkpointer=MemorySaver(),
    interrupt_before=['light'],  # pause before this node
)

# Run until the interrupt point
config = {'configurable': {'thread_id': 'session-001'}}
app.invoke(INITIAL_STATE, config=config)
# Graph pauses after edgeworth runs, before light runs.

# Inspect or modify state:
current_state = app.get_state(config)
print(current_state.values['code'])  # Edgeworth's output

# Optionally modify the state before resuming:
app.update_state(config, {'last_critique': 'Human override: simplify the abstractions'})

# Resume:
app.invoke(None, config=config)  # None = continue from checkpoint
```

## 3. Streaming — Watch Each Turn As It Happens

CCA prints output inside each agent function. LangGraph's `stream()` emits events at the graph level, giving you a single place to observe all state changes.

```python
for chunk in cca_app.stream(INITIAL_STATE, stream_mode='updates'):
    node_name = list(chunk.keys())[0]
    delta = chunk[node_name]
    if 'code' in delta:
        print(f'[{node_name.upper()}] wrote {len(delta["code"])} chars of code')
    if 'last_critique' in delta:
        print(f'[{node_name.upper()}] critique: {delta["last_critique"][:100]}...')
```

## 4. Context Trimming — Built-in, Not Hand-Rolled

CCA's `context_trimmer.py` is a custom implementation of a common problem. LangGraph provides `trim_messages()` from `langchain_core` that integrates with the `add_messages` reducer:

```python
from langchain_core.messages import trim_messages

def edgeworth_node(state: CCAState) -> dict:
    # Trim history before building the message list
    trimmed_history = trim_messages(
        list(state.get('edgeworth_history', [])),
        max_tokens=8_000,
        token_counter=ChatAnthropic(model=AGENT_MODEL),
        strategy='last',          # keep the most recent messages
        include_system=False,
    )
    # Use trimmed_history in the call instead of state['edgeworth_history']
    ...
```


---
## Summary

### LangChain vs raw SDK

| Task | Raw SDK (CCA) | LangChain |
|---|---|---|
| Call a model | `client.messages.create(system=[...], messages=[...])` | `model.invoke([SystemMessage(...), HumanMessage(...)])` |
| Build a prompt | f-string in `build_context_package()` | `ChatPromptTemplate.from_messages([...])` |
| Chain steps | Manual: call function, pass result | `prompt \| model \| parser` (LCEL) |
| Parse output | Regex on `<code>` / `<critique>` tags | `model.with_structured_output(PydanticSchema)` |
| Switch providers | Rewrite all API calls | Change `ChatAnthropic` to `ChatOpenAI` — same interface |

### LangGraph vs hand-rolled orchestration

| Task | Hand-rolled (CCA) | LangGraph |
|---|---|---|
| Agent loop | `for round_num in range(...)` in `orchestrator.py` | Cyclic graph edges |
| State | Instance variables on `Orchestrator` | `TypedDict` flowing through nodes |
| History accumulation | Manual list append + `context_trimmer.py` | `add_messages` reducer + `trim_messages()` |
| Routing logic | `if round_num >= 3: ...` inside agent function | Conditional edge routing function |
| Session persistence | `WorkspaceManager` writes files to disk | `SqliteSaver` / `MemorySaver` checkpointer |
| Human feedback | `run_feedback_mode()` custom interactive loop | `interrupt_before=[]` + `update_state()` |
| Observability | `print()` inside agent functions | `stream(mode='updates')` at graph level |

### When to use which

- **Raw SDK**: educational projects, complete control, no abstraction overhead. CCA is a perfect example — the hand-rolled version IS the teaching material.
- **LangChain**: when you need provider flexibility, want composable chains, or need structured output without writing parsers.
- **LangGraph**: when you have a multi-agent loop with shared state, need session persistence, or want human-in-the-loop. The more complex your orchestration, the more LangGraph earns its abstraction cost.
